# BenchMark comprasion

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import json

sibling_b_path = "logs/sibling_b.csv"
metrics_log    = "logs/metrics_log.csv"
output_json    = "results/benchmark_results.json"


def load_encode_data(path=sibling_b_path):
    df = pd.read_csv(path)
    print("Sibling B data loaded successfully.")

    all_chars = set()
    for col in df.columns:
        all_chars.update(df[col].astype(str).unique())
    vocab      = sorted(all_chars)
    char_to_id = {ch: i for i, ch in enumerate(vocab)}

    y_true       = df["Readable_Actual_Target"].astype(str).map(char_to_id).values
    y_pred_knn   = df["KNN_Prediction"].astype(str).map(char_to_id).values
    y_pred_dt    = df["DT_Prediction"].astype(str).map(char_to_id).values
    y_pred_label = df["LabelProp_Prediction"].astype(str).map(char_to_id).values

    np.save("results/y_true.npy",       y_true)
    np.save("results/y_pred_knn.npy",   y_pred_knn)
    np.save("results/y_pred_dt.npy",    y_pred_dt)
    np.save("results/y_pred_label.npy", y_pred_label)

    return y_true, y_pred_knn, y_pred_dt, y_pred_label


def load_neural_net_loss(csv_path=metrics_log):
    df = pd.read_csv(csv_path)
    # BUG FIX: added 4 decimal places to round() so values are not truncated to integers
    final_train_loss = round(float(df["train_loss"].iloc[-1]), 4)
    final_val_loss   = round(float(df["val_loss"].iloc[-1]),   4)
    best_val_loss    = round(float(df["val_loss"].min()),       4)
    best_step        = int(df.loc[df["val_loss"].idxmin(), "step"])
    total_steps      = int(df["step"].iloc[-1])

    accuracy = 0.2058  # Hardcoded — replace with Harsha's exported value when available
    f1       = 0.2564

    print("\n==================================================")
    print("  NEURAL NET Stats from metrics_log.csv")
    print("==================================================")
    print(f"  Total Training Steps : {total_steps}")
    print(f"  Final Train Loss     : {final_train_loss}")
    print(f"  Final Val Loss       : {final_val_loss}")
    print(f"  Best Val Loss        : {best_val_loss}  (at step {best_step})")
    print(f"  Accuracy             : {accuracy}")
    print(f"  F1 Score             : {f1}\n")

    return {"accuracy": accuracy, "f1": f1}


if __name__ == "__main__":
    all_results = []

    nn_stats = load_neural_net_loss()
    y_true, y_pred_knn, y_pred_dt, y_pred_label = load_encode_data()

    models = {
        "KNN Classifier (Sibling B)":   y_pred_knn,
        "Decision Tree (Sibling B)":    y_pred_dt,
        "Label Propagation (Sibling A)": y_pred_label,
    }

    print("\nModel Evaluation Results")
    print("-" * 50)
    for name, predictions in models.items():
        acc = accuracy_score(y_true, predictions)
        f1  = f1_score(y_true, predictions, average="macro", zero_division=0)
        print(f"  {name} -> Accuracy: {acc:.4f} | F1: {f1:.4f}")
        all_results.append({"name": name, "accuracy": round(acc, 4), "f1": round(f1, 4)})

    # Append Neural Net entry
    all_results.append({"name": "PyTorch Neural Net", "accuracy": nn_stats["accuracy"], "f1": nn_stats["f1"]})

    with open(output_json, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved benchmark results -> {output_json}")


# plotting the graph

In [ ]:
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score


def plot_training(csv_path="logs/metrics_log.csv"):
    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 5))
    plt.plot(df["step"], df["train_loss"], label="Train Loss",      color="blue")
    plt.plot(df["step"], df["val_loss"],   label="Validation Loss", color="orange")
    plt.title("Neural Network Training Loss")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/training_loss.png", dpi=150)
    print("Training loss chart saved -> results/training_loss.png")
    plt.close()


def plot_accuracy(sibling_path="logs/sibling_b.csv", benchmark_json="results/benchmark_results.json"):
    df    = pd.read_csv(sibling_path)
    y_true = df["Readable_Actual_Target"]

    knn_acc   = accuracy_score(y_true, df["KNN_Prediction"])
    dt_acc    = accuracy_score(y_true, df["DT_Prediction"])
    label_acc = accuracy_score(y_true, df["LabelProp_Prediction"])

    # BUG FIX: include PyTorch Neural Net as the 4th bar (was missing)
    nn_acc = 0.2058  # Hardcoded — replace with Harsha's value when available

    models = ["KNN", "Decision Tree", "Label Propagation", "PyTorch Neural Net"]
    scores = [knn_acc, dt_acc, label_acc, nn_acc]
    colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]

    plt.figure(figsize=(9, 5))
    bars = plt.bar(models, scores, color=colors)
    plt.title("Overall Model Accuracy Comparison")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.0)

    # Add value labels on top of each bar
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{score:.4f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig("results/overall_accuracy.png", dpi=150)
    print("Overall accuracy chart saved -> results/overall_accuracy.png")
    plt.close()


if __name__ == "__main__":
    print("Generating plots...")
    plot_training()
    plot_accuracy()
    print("Done.")


# Performance forecaster

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# -- Hyperparameter-based forecasting (run_logs.csv) --------------------------
def load_run_logs(path="run_logs.csv"):
    df = pd.read_csv(path)
    feature_cols = ["vocab_size", "embedding_dim", "context_block_size", "batch_size", "learning_rate"]
    X      = df[feature_cols].values
    y_loss = df["val_loss"].values
    y_time = df["train_time_seconds"].values
    print(f"Loaded {len(df)} hyperparameter run records from {path}")
    return X, y_loss, y_time, feature_cols


# -- Step-based forecasting (metrics_log.csv) ---------------------------------
def load_training_steps(metrics_path="logs/metrics_log.csv"):
    df = pd.read_csv(metrics_path)
    x = df["step"].values.reshape(-1, 1)
    y = df["val_loss"].values
    print(f"Loaded {len(df)} training step records from {metrics_path}")
    return x, y


if __name__ == "__main__":
    print("=" * 60)
    print("  PERFORMANCE FORECASTER")
    print("=" * 60)

    # -- Part 1: Forecast val_loss & train_time from hyperparameters ----------
    print("\n[1] Hyperparameter -> Val Loss / Train Time Forecasting")
    X, y_loss, y_time, feature_cols = load_run_logs()

    # BUG FIX: correct variable names (was 'model' which was undefined)
    model_linear = LinearRegression()
    model_linear.fit(X, y_loss)

    # NEW: Polynomial Regression (degree=2)
    poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
    poly_model.fit(X, y_loss)

    # NEW: KNN Regressor
    knn_reg = KNeighborsRegressor(n_neighbors=3)
    knn_reg.fit(X, y_loss)

    # Decision Tree Regressor for val_loss
    decision_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
    decision_tree.fit(X, y_loss)

    # Train time regressor (Linear)
    time_model = LinearRegression()
    time_model.fit(X, y_time)

    # Example: predict for new hypothetical hyperparameter sets
    new_configs = pd.DataFrame([
        {"vocab_size": 150, "embedding_dim": 64,  "context_block_size": 16, "batch_size": 64,  "learning_rate": 0.001},
        {"vocab_size": 200, "embedding_dim": 128, "context_block_size": 32, "batch_size": 128, "learning_rate": 0.0005},
    ])
    X_new = new_configs.values

    print(f"\n  {'Config':<8} | {'Linear Reg':<12} | {'Poly Reg':<12} | {'KNN Reg':<12} | {'DT Reg'}")
    print("  " + "-" * 68)
    for i, (lin, poly, knn_p, dt) in enumerate(zip(
            model_linear.predict(X_new),
            poly_model.predict(X_new),
            knn_reg.predict(X_new),
            decision_tree.predict(X_new))):
        print(f"  Config {i+1:<3} | {lin:<12.4f} | {poly:<12.4f} | {knn_p:<12.4f} | {dt:.4f}")

    print("\n  Estimated train times (seconds):")
    for i, t in enumerate(time_model.predict(X_new)):
        print(f"    Config {i+1}: {t:.1f}s  (~{t/60:.1f} min)")

    # -- Part 2: Future step forecasting from training curve ------------------
    print("\n[2] Training Step -> Val Loss Forecasting")
    x_train, y_train = load_training_steps()

    # BUG FIX: renamed to model_linear_steps to avoid shadowing
    model_linear_steps = LinearRegression()
    model_linear_steps.fit(x_train, y_train)

    decision_tree_steps = DecisionTreeRegressor(max_depth=3, random_state=42)
    decision_tree_steps.fit(x_train, y_train)

    # NEW: Polynomial on steps
    poly_steps = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
    poly_steps.fit(x_train, y_train)

    future_steps = np.array([40000, 50000, 75000, 100000]).reshape(-1, 1)

    print(f"\n  {'Future Step':<14} | {'Linear Reg':<12} | {'Poly Reg':<12} | {'DT Reg'}")
    print("  " + "-" * 58)
    for step, lin, poly, dt in zip(
            future_steps,
            model_linear_steps.predict(future_steps),
            poly_steps.predict(future_steps),
            decision_tree_steps.predict(future_steps)):
        print(f"  {step[0]:<14,} | {lin:<12.4f} | {poly:<12.4f} | {dt:.4f}")

    # -- Part 3: Plot regression curves ---------------------------------------
    print("\n[3] Saving regression forecast chart...")
    x_plot = np.linspace(0, 100000, 300).reshape(-1, 1)

    plt.figure(figsize=(10, 5))
    plt.scatter(x_train, y_train, s=8, alpha=0.4, label="Actual Val Loss", color="gray")
    plt.plot(x_plot, model_linear_steps.predict(x_plot), label="Linear Regression",      color="blue")
    plt.plot(x_plot, poly_steps.predict(x_plot),          label="Polynomial Regression",  color="green")
    plt.plot(x_plot, decision_tree_steps.predict(x_plot), label="Decision Tree Regressor",color="red", linestyle="--")
    plt.title("Val Loss Forecast: Regression Models")
    plt.xlabel("Training Step")
    plt.ylabel("Predicted Val Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/regression_forecast.png", dpi=150)
    print("Regression forecast chart saved -> results/regression_forecast.png")
    plt.close()

    print("\nPerformance Forecaster complete.")
